In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense

In [2]:
df = pd.read_csv(r'D:\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\AutoEncoder\NASA Turbofan ENgine\NASA TurboFan Engine.csv')

In [3]:
print(df.head())
print(df.shape)

   unit  cycle  setting1  setting2  setting3  sensor1  sensor2  sensor3  \
0     1      1   -0.0007   -0.0004     100.0   518.67   641.82  1589.70   
1     1      2    0.0019   -0.0003     100.0   518.67   642.15  1591.82   
2     1      3   -0.0043    0.0003     100.0   518.67   642.35  1587.99   
3     1      4    0.0007    0.0000     100.0   518.67   642.35  1582.79   
4     1      5   -0.0019   -0.0002     100.0   518.67   642.37  1582.85   

   sensor4  sensor5  ...  sensor14  sensor15  sensor16  sensor17  sensor18  \
0  1400.60    14.62  ...   8138.62    8.4195      0.03       392      2388   
1  1403.14    14.62  ...   8131.49    8.4318      0.03       392      2388   
2  1404.20    14.62  ...   8133.23    8.4178      0.03       390      2388   
3  1401.87    14.62  ...   8133.83    8.3682      0.03       392      2388   
4  1406.22    14.62  ...   8133.80    8.4294      0.03       393      2388   

   sensor19  sensor20  sensor21  RUL  Fault_Label  
0     100.0     39.06   23.4

In [5]:
X = df.drop(["unit", "cycle", "RUL", "Fault_Label"], axis=1)
y = df["Fault_Label"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train[y_train == 0]

In [9]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

C:\tf-env\Lib\site-packages\sklearn\utils\validation.py:2823: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [11]:
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

In [ ]:
# NASA TURBOFAN ENGINE FAULT DETECTION
# USING LSTM AUTOENCODER

# ==========================================
# 1. IMPORT REQUIRED LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense


# ==========================================
# 2. LOAD THE DATASET
# ==========================================

data = pd.read_csv("NASA TurboFan Engine.csv")

print("Dataset Shape:", data.shape)
print(data.head())


# ==========================================
# 3. PREPROCESS THE DATA
# ==========================================

# Save actual fault labels
y = data["Fault_Label"].values

# Select sensor columns
sensor_columns = [col for col in data.columns if col.startswith("sensor")]

# Extract sensor readings
X = data[sensor_columns]

# Fill missing values
X = X.fillna(X.mean())

# Normalize data
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print("Input Shape:", X.shape)


# ==========================================
# 4. CREATE TIME WINDOWS
# ==========================================

window_size = 10

X_window = []
y_window = []

for unit in data["unit"].unique():

    # Get readings of one engine
    unit_indices = data.index[data["unit"] == unit].tolist()

    X_unit = X[unit_indices]
    y_unit = y[unit_indices]

    for i in range(len(X_unit) - window_size + 1):

        X_window.append(X_unit[i:i + window_size])

        # Label window as fault if it contains any fault
        y_window.append(int(np.any(y_unit[i:i + window_size] == 1)))

X_window = np.array(X_window)
y_window = np.array(y_window)

print("Window Shape:", X_window.shape)


# ==========================================
# 5. SPLIT DATA INTO TRAINING AND TESTING
# ==========================================

split = int(0.8 * len(X_window))

X_train = X_window[:split]
X_test = X_window[split:]

y_train = y_window[:split]
y_test = y_window[split:]

# Keep only healthy windows for training
X_train = X_train[y_train == 0]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


# ==========================================
# 6. BUILD LSTM AUTOENCODER
# ==========================================

input_layer = Input(
    shape=(window_size, X_train.shape[2])
)

# Encoder
encoded = LSTM(32, activation="relu")(input_layer)

# Decoder
decoded = RepeatVector(window_size)(encoded)

decoded = LSTM(
    32,
    activation="relu",
    return_sequences=True
)(decoded)

decoded = TimeDistributed(
    Dense(X_train.shape[2])
)(decoded)

autoencoder = Model(input_layer, decoded)


# ==========================================
# 7. COMPILE THE MODEL
# ==========================================

autoencoder.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)


# ==========================================
# 8. TRAIN THE MODEL
# ==========================================

autoencoder.fit(
    X_train,
    X_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# ==========================================
# 9. CALCULATE RECONSTRUCTION ERROR
# ==========================================

# Training reconstruction error
train_reconstructed = autoencoder.predict(X_train)

train_error = np.mean(
    np.square(X_train - train_reconstructed),
    axis=(1, 2)
)

# Testing reconstruction error
test_reconstructed = autoencoder.predict(X_test)

test_error = np.mean(
    np.square(X_test - test_reconstructed),
    axis=(1, 2)
)


# ==========================================
# 10. SET THRESHOLD
# ==========================================

threshold = np.percentile(train_error, 95)


# ==========================================
# 11. DETECT FAULTS
# ==========================================

predictions = (
    test_error > threshold
).astype(int)

print("\nThreshold:", threshold)

print(
    "Number of Faults Detected:",
    np.sum(predictions)
)


# ==========================================
# 12. SHOW RESULTS
# ==========================================

results = pd.DataFrame({
    "Actual_Fault": y_test,
    "Reconstruction_Error": test_error,
    "Predicted_Fault": predictions
})

print("\nResults:")
print(results.head(10))


# ==========================================
# 13. EVALUATE MODEL
# ==========================================

from sklearn.metrics import confusion_matrix, classification_report

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")

print(classification_report(
    y_test,
    predictions,
    target_names=["Normal", "Fault"],
    zero_division=0
))